# Day 64: Neural Networks with Keras

## Install Note
Everything up to this point has used scikit-learn-style models (trees, SVMs, clustering), which are great for tabular data but limited when you need to learn very flexible, layered representations — e.g. from images, text, or audio. Neural networks, and Keras specifically (a high-level API bundled with TensorFlow), let you build models as stacks of simple mathematical layers that are trained together via gradient descent. This notebook is the first step into deep learning: a basic fully-connected ("dense") network trained on the familiar Iris dataset, so you can focus on the new mechanics (layers, compiling, epochs) without also having to learn a new dataset. If TensorFlow isn't already installed, run the pip command below once in your environment.

In [ ]:
# pip install tensorflow

## Preparing Data
Neural networks are sensitive to feature scale in the same way distance-based ML methods are — unscaled features can make training slow or unstable — so we standardize `X` just like we did for PCA/clustering. The bigger difference from scikit-learn workflows is the target: scikit-learn classifiers happily take integer class labels (0, 1, 2), but a neural network's final layer for multi-class classification outputs one probability per class, so the target needs to be "one-hot encoded" — each label becomes a vector with a 1 in the position of the correct class and 0s elsewhere (e.g. class 1 of 3 becomes `[0, 1, 0]`). `to_categorical(y)` does exactly this conversion, producing `y_cat`, which we then split into train/test sets as usual.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

X, y = load_iris(return_X_y=True)
# Standardize features -- neural network training (gradient descent) converges better on scaled inputs
X = StandardScaler().fit_transform(X)
# One-hot encode labels: e.g. class 1 -> [0, 1, 0], needed for the softmax output layer below
y_cat = to_categorical(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)

## Building a Sequential Model
`keras.Sequential` is the simplest way to define a neural network: you list its layers in order, and data flows through them top to bottom. Each `Dense` layer is a fully-connected layer — every input is connected to every neuron in that layer via a learned weight — followed by an activation function that introduces non-linearity (without it, stacking layers would collapse into one big linear function no matter how many layers you add). Here the network has 4 inputs (the Iris features), a hidden layer of 16 neurons with `relu` activation, a second hidden layer of 8 neurons also with `relu`, and a final output layer of 3 neurons (one per Iris species) with `softmax` activation, which converts the raw outputs into a probability distribution that sums to 1 across the 3 classes. `model.summary()` prints each layer's shape and parameter count, which is a good sanity check that the architecture matches what you intended before you spend time training it.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# A simple feedforward network: 4 input features -> 16 ReLU neurons -> 8 ReLU neurons -> 3-class softmax output
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(4,)),  # hidden layer 1, ReLU adds non-linearity
    layers.Dense(8, activation='relu'),                       # hidden layer 2, further combines features
    layers.Dense(3, activation='softmax')                     # output layer: 3 class probabilities summing to 1
])
model.summary()

## Compiling & Training
Before a Keras model can be trained it must be "compiled" — this is where you specify the optimizer (the algorithm that updates the network's weights, here `adam`, a popular adaptive variant of gradient descent), the loss function the optimizer tries to minimize (`categorical_crossentropy`, the standard loss for multi-class classification with one-hot encoded, softmax-output targets), and any extra metrics to track during training (`accuracy`, which isn't used for the actual weight updates but is reported for you to monitor). `fit()` then trains the network: an "epoch" is one full pass through the entire training set, `batch_size=8` means weights are updated after every 8 samples rather than one at a time or all at once, and `validation_split=0.2` holds out 20% of the training data (not the test set) to monitor how well the model generalizes during training itself. Finally, `model.evaluate` on the untouched test set reports the loss and accuracy on data the model never saw at all, which is the number that actually matters for judging the model.

In [ ]:
# adam optimizer + categorical_crossentropy loss is the standard combo for multi-class classification
# with one-hot targets and a softmax output layer
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# epochs=50 full passes over the training data; batch_size=8 samples per weight update;
# validation_split=0.2 holds out part of the training data (not the test set) to monitor generalization
history = model.fit(X_train, y_train, epochs=50, batch_size=8, validation_split=0.2, verbose=0)
# Evaluate on the held-out test set; index [1] pulls out accuracy (index [0] would be the loss)
print('Test accuracy:', model.evaluate(X_test, y_test, verbose=0)[1])